# 06 — Backpropagation, MLP e problema XOR

**EP · Data Science Knowledge Base | Deep Learning | Notebook 06/20**


**Pré-requisitos:** Python, NumPy, fundamentos de aprendizado supervisionado; JupyterLab; TensorFlow/Keras apenas nos laboratórios indicados.

**Escopo de validação:** os exemplos básicos podem ser reproduzidos sem TensorFlow; laboratórios opcionais precisam de validação local independente.

---

**Deep Learning · Capítulo de estudo independente**

> Material didático autoral · estudo independente · exemplos locais e laboratórios opcionais

**Referência curricular:** `material histórico de referência`. O material original foi preservado; este capítulo é uma reconstrução didática independente.

### Materiais complementares

- [Guia Técnico](./reference_materials/06_backpropagation_mlp_xor.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_backpropagation_mlp_xor.pdf)


## Como estudar este capítulo

1. Leia a teoria e deduza as expressões apresentadas.
2. Execute as células do **núcleo obrigatório** em um kernel limpo.
3. Confira a interpretação e responda ao exercício antes de abrir a solução.
4. Se tiver TensorFlow e Keras compatíveis, execute o **laboratório opcional**.

**Pré-requisitos transversais:** Python (01), Estatística (03), preparação/validação de ML (05); aplicações textuais remetem à PLN (06). Não há necessidade de repetir integralmente esses cursos.

## Objetivos de aprendizagem

- Aplicar regra da cadeia e mostrar por que uma camada oculta representa XOR.
- Relacionar a técnica à sua função de perda, representação e protocolo de avaliação.
- Explicar limitações e formular um teste reprodutível.

## 1. Fundamentação conceitual

Retropropagação propaga derivadas da perda da saída para as camadas iniciais. É um algoritmo para calcular gradientes; o otimizador decide como atualizar parâmetros. Ativações não lineares são necessárias para representar relações como XOR.

**Fonte histórica de apoio:** notas do curso original, consultadas apenas como referência de contexto; as explicações e exemplos desta edição são independentes.

**Questões críticas:** quais pressupostos são necessários? O que a função aprende? Qual métrica indica utilidade real? Que tipo de vazamento ou má interpretação invalidaria a conclusão?

## 2. Matemática essencial

**Expressão esquemática:** `\delta _l=(W_(l+1) \delta _(l+1)) ⊙ f′(z_l)`

A expressão acima resume a relação central. Para a derivação, identifique dimensões, hipóteses e operações; não interprete notação compacta como uma implementação completa.

## 3. Experimento obrigatório — NumPy e/ou scikit-learn

Este experimento é propositalmente pequeno: **executa sem GPU, internet ou serviço externo**. Quando há treino de baseline, não interpretá-lo como evidência da qualidade de uma rede neural.

In [1]:
import numpy as np
X=np.array([[0,0],[0,1],[1,0],[1,1]],float)
# Representação explícita por duas unidades ocultas: OR e NAND
h1=(X.sum(axis=1)>=1).astype(int)
h2=(X.sum(axis=1)<2).astype(int)
yhat=(h1+h2==2).astype(int)
print('OR:',h1.tolist(),'NAND:',h2.tolist(),'XOR:',yhat.tolist())
assert np.array_equal(yhat,[0,1,1,0])

OR: [0, 1, 1, 1] NAND: [1, 1, 1, 0] XOR: [0, 1, 1, 0]


### Interpretação e limitações

Compare o resultado obtido com a formulação matemática. Confirme shapes, escala e hipótese de modelagem. **Um experimento sintético valida uma operação, não substitui avaliação em dados reais.** Ao treinar modelos, separe dados e ajuste transformações apenas no treino; preserve um conjunto final de teste.

## 4. Laboratório neural opcional — TensorFlow / Keras 3

Esta seção demonstra **APIs reais**, mas depende de versões compatíveis e pode exigir mais recursos que o núcleo. Não é necessária para executar o notebook básico. Não se deve confundir um teste de chamada de API com avaliação experimental de desempenho. Se o pacote TensorFlow não estiver disponível, será exibido um aviso de não execução.

In [2]:
# Laboratório opcional: execute após instalar e configurar TensorFlow/Keras compatíveis.
# Se TensorFlow estiver ausente, o exemplo é ignorado com aviso explícito.
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    print('Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.')
else:
    import keras
    from keras import layers
    net=keras.Sequential([keras.Input(shape=(2,)),layers.Dense(8,activation='tanh'),layers.Dense(1,activation='sigmoid')])
    net.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    import numpy as np
    XX=np.array([[0,0],[0,1],[1,0],[1,1]],dtype='float32'); yy=np.array([0,1,1,0],dtype='float32')
    net.fit(XX,yy,epochs=80,batch_size=4,verbose=0)
    print(np.round(net.predict(XX,verbose=0).ravel(),3))

Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.


## 5. Exercício de fixação

**Questão:** Por que ativação puramente linear em todas as camadas não resolve XOR?

<details>
<summary>Confira a solução comentada</summary>

**Solução:** A composição de funções afins permanece afim; não cria fronteiras não lineares.

**Verificação:** explique o resultado usando notação, shapes e consequências para generalização.
</details>

## 6. Erros comuns e boas práticas

- Confundir ajuste no treino com generalização para dados não vistos.
- Reportar somente uma métrica ou uma execução aleatória.
- Ignorar shape, tipo de variável e convenção do backend.
- Executar exemplos com pesos baixados sem registrar versão, licença e procedência.
- Confundir demonstração matemática com validação de um modelo de produção.

## 7. Resumo de consulta rápida

**Ideia central:** Retropropagação propaga derivadas da perda da saída para as camadas iniciais.

**Expressão:** `δ_l=(W_(l+1) δ_(l+1)) ⊙ f′(z_l)`.

**Lembrete:** começar por baseline; respeitar separação de dados; documentar decisões; avaliar limitações.

**Ponte com outros cursos:** ML I (05) para métricas e validação; PLN (06) para modelos de linguagem; projetos multidisciplinares ficam reservados à fase posterior.

**Fontes de atualização (consultar quando necessário):** [Keras 3](https://keras.io/guides/), [TensorFlow](https://www.tensorflow.org/guide), [scikit-learn](https://scikit-learn.org/stable/user_guide.html).

## Laboratório aplicado revisado — Retropropagação numérica em uma MLP para XOR

A dedução completa exige derivadas de cada camada, propagação reversa e atualização distinta dos pesos. A versão NumPy explicita todas essas etapas.

**Procedimento:** execute a próxima célula com o núcleo declarado em `requirements.txt`, confira os `assert` e interprete os resultados. O experimento é independente dos exemplos anteriores.

In [3]:
import numpy as np
X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y = np.array([[0],[1],[1],[0]], dtype=float)
rng = np.random.default_rng(17)
W1 = rng.normal(0, 0.7, (2, 8)); b1 = np.zeros((1,8))
W2 = rng.normal(0, 0.7, (8, 1)); b2 = np.zeros((1,1))
def sig(z): return 1/(1+np.exp(-np.clip(z,-30,30)))
losses=[]
for epoch in range(3000):
    a1 = np.tanh(X @ W1 + b1)
    p = sig(a1 @ W2 + b2)
    loss = -np.mean(y*np.log(p+1e-9)+(1-y)*np.log(1-p+1e-9))
    losses.append(loss)
    d2=(p-y)/len(X)
    dW2=a1.T@d2; db2=d2.sum(axis=0,keepdims=True)
    d1=(d2@W2.T)*(1-a1**2)
    dW1=X.T@d1; db1=d1.sum(axis=0,keepdims=True)
    W2-=0.8*dW2; b2-=0.8*db2; W1-=0.8*dW1; b1-=0.8*db1
pred=(sig(np.tanh(X@W1+b1)@W2+b2)>=.5).astype(int).ravel()
assert np.array_equal(pred, y.ravel().astype(int)), pred
assert losses[-1]<losses[0]
print('Predições XOR:',pred.tolist(),'perda inicial/final:',round(losses[0],4),round(losses[-1],4))

Predições XOR: [0, 1, 1, 0] perda inicial/final: 0.8072 0.0006


### Interpretação, limites e verificação

Treino na própria tabela-verdade demonstra capacidade de ajuste, não generalização. Verificar gradientes e inicialização; XOR tem somente quatro casos.

**Checklist:** conferir entradas e saídas, sementes, inexistência de uso dos dados de teste durante ajuste e distinção entre uma demonstração sintética e desempenho em aplicações reais.